# 04 - Fine-tune Cross-Encoder Re-ranker

Train a Vietnamese CSKH re-ranker using CrossEncoder.

### Purpose
Re-ranker sits **after** FAISS retrieval — it takes top-K candidates from embedding
search and re-scores them with a more accurate cross-attention model.

### Advanced techniques:
- **Cross-Encoder** (cross-attention, more accurate than bi-encoder)
- **Hard negative mining** from FAISS retrieval results
- **Balanced positive/negative pairs** for training stability
- **Google Drive save**

### Requirements
- Colab with **T4 GPU** (~1 compute unit)

### Output
- Fine-tuned CrossEncoder model for backend `reranker.py`

In [ ]:
# ⚠️ SAU KHI CHẠY XONG → Runtime → Restart session → rồi chạy từ Cell 2
!pip install -q \
    "huggingface_hub>=0.25" \
    "sentence-transformers>=3.0" \
    "datasets>=3.0"

print('✅ Done! Bây giờ hãy: Runtime → Restart session → rồi chạy từ Cell 2')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random
import torch
import numpy as np

DRIVE_DIR = '/content/drive/MyDrive/vani-copilot'
os.makedirs(DRIVE_DIR, exist_ok=True)
random.seed(42)

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Load knowledge base + training data
from pathlib import Path

# Verify files exist
for f in ['train.jsonl', 'faq.txt', 'policies.txt', 'products.txt']:
    p = os.path.join(DRIVE_DIR, f)
    ok = os.path.exists(p)
    sz = os.path.getsize(p) if ok else 0
    print(f'  {"✅" if ok else "❌"} {f} ({sz:,} bytes)')
    if not ok:
        raise FileNotFoundError(
            f'{f} not found in {DRIVE_DIR}\n'
            '👉 Upload colab-upload/ files to MyDrive/vani-copilot/'
        )

knowledge_docs = []
for fname in ['policies.txt', 'faq.txt', 'products.txt']:
    p = Path(DRIVE_DIR) / fname
    text = p.read_text(encoding='utf-8')
    paragraphs = [para.strip() for para in text.split('\n\n') if para.strip() and len(para.strip()) > 20]
    knowledge_docs.extend(paragraphs)
    print(f'{fname}: {len(paragraphs)} paragraphs')

with open(os.path.join(DRIVE_DIR, 'train.jsonl'), 'r', encoding='utf-8') as f:
    train_data = [json.loads(line) for line in f]
print(f'Train data: {len(train_data)} samples')

print(f'\nTotal knowledge docs: {len(knowledge_docs)}')

In [ ]:
# Step 1: Use fine-tuned embedding (or base) to generate hard negatives
from sentence_transformers import SentenceTransformer

embed_path = os.path.join(DRIVE_DIR, 'embedding-finetuned')
if os.path.exists(embed_path):
    bi_encoder = SentenceTransformer(embed_path)
    print(f'Loaded fine-tuned embedding from Drive')
else:
    bi_encoder = SentenceTransformer('intfloat/multilingual-e5-base')
    print('Using base embedding (fine-tuned not found on Drive)')

In [ ]:
# Step 2: Build corpus from knowledge base + assistant answers
corpus = list(knowledge_docs)

for item in train_data:
    for msg in item['messages']:
        if msg['role'] == 'assistant' and len(msg['content'].strip()) > 20:
            corpus.append(msg['content'].strip())

# Deduplicate
corpus = list(set(corpus))
print(f'Corpus size: {len(corpus)}')

# Encode all corpus
corpus_embeddings = bi_encoder.encode(
    [f'passage: {d}' for d in corpus],
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64,
)
print(f'Corpus encoded: {corpus_embeddings.shape}')

In [ ]:
# Step 3: Generate (query, passage, label) with hard negatives
# Positive = user question + correct assistant answer (label=1)
# Hard negative = user question + top-retrieved but WRONG doc (label=0)

import gc
from sentence_transformers import InputExample

# Extract all (query, positive) pairs first
pairs = []
for item in train_data:
    msgs = item['messages']
    for i in range(len(msgs) - 1):
        if msgs[i]['role'] == 'user' and msgs[i+1]['role'] == 'assistant':
            query = msgs[i]['content'].strip()
            positive = msgs[i+1]['content'].strip()
            if len(query) >= 10 and len(positive) >= 15:
                pairs.append((query, positive))

MAX_PAIRS = 5000
if len(pairs) > MAX_PAIRS:
    random.shuffle(pairs)
    pairs = pairs[:MAX_PAIRS]
print(f'Query-answer pairs: {len(pairs)}')

# Batch encode all queries at once (MUCH faster than one-by-one)
all_queries = [f'query: {q}' for q, _ in pairs]
print('Encoding queries (batch)...')
query_embeddings = bi_encoder.encode(
    all_queries, normalize_embeddings=True,
    batch_size=64, show_progress_bar=True,
)

# Generate train samples with hard negatives
train_samples = []
for i, (query, positive) in enumerate(pairs):
    train_samples.append(InputExample(texts=[query, positive], label=1.0))

    scores = query_embeddings[i] @ corpus_embeddings.T
    top_indices = np.argsort(scores)[-10:][::-1]
    for idx in top_indices:
        candidate = corpus[idx]
        if candidate != positive and len(candidate) > 15:
            train_samples.append(InputExample(texts=[query, candidate], label=0.0))
            break

random.shuffle(train_samples)
gc.collect()

n_pos = sum(1 for s in train_samples if s.label > 0.5)
n_neg = len(train_samples) - n_pos
print(f'Total samples: {len(train_samples)}')
print(f'Positive: {n_pos}, Negative: {n_neg}')
print(f'Ratio: {n_pos / max(n_neg, 1):.2f}')

In [ ]:
# Step 4: Train CrossEncoder
# Dùng cross-encoder đã pre-train cho ranking (thay vì xlm-roberta-base chưa biết gì)
import gc
from sentence_transformers import CrossEncoder
from torch.utils.data import DataLoader
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

gc.collect()
torch.cuda.empty_cache()

cross_encoder = CrossEncoder(
    'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1',
    num_labels=1,
    max_length=256,
)

# Split for eval
eval_size = min(500, len(train_samples) // 10)
eval_samples = train_samples[:eval_size]
fit_samples = train_samples[eval_size:]

train_dataloader = DataLoader(fit_samples, shuffle=True, batch_size=16)

# Build evaluator (compatible with sentence-transformers v3+)
eval_sentences1 = [s.texts[0] for s in eval_samples]
eval_sentences2 = [s.texts[1] for s in eval_samples]
eval_labels = [int(s.label) for s in eval_samples]

try:
    evaluator = CEBinaryClassificationEvaluator(
        sentences1=eval_sentences1,
        sentences2=eval_sentences2,
        labels=eval_labels,
        name='reranker_eval',
    )
except TypeError:
    evaluator = CEBinaryClassificationEvaluator(
        sentence_pairs=list(zip(eval_sentences1, eval_sentences2)),
        labels=eval_labels,
        name='reranker_eval',
    )

print(f'Train: {len(fit_samples)} samples')
print(f'Eval: {len(eval_samples)} samples')
print(f'Starting training...')

cross_encoder.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=5,
    warmup_steps=100,
    evaluation_steps=200,
    output_path='./reranker-finetuned',
    show_progress_bar=True,
)

print('✅ Re-ranker training complete!')

In [ ]:
# Step 5: Test re-ranker
test_queries = [
    ('đổi trả hàng bị lỗi', 'Hàng bị lỗi sản xuất được đổi miễn phí trong 7 ngày'),
    ('đổi trả hàng bị lỗi', 'Áo croptop mới về chất liệu cotton 100%'),
    ('ship bao lâu', 'Giao hàng nội thành 1-2 ngày, ngoại thành 3-5 ngày'),
    ('ship bao lâu', 'Mã giảm giá VANI50 giảm 50k cho đơn từ 500k'),
]

for q, d in test_queries:
    score = cross_encoder.predict([(q, d)])[0]
    label = 'RELEVANT' if score > 0.5 else 'NOT relevant'
    print(f'[{score:.3f}] {label} | Q: "{q}" | D: "{d[:60]}..."')

In [ ]:
# Save to Drive
import shutil

save_path = os.path.join(DRIVE_DIR, 'reranker-finetuned')
if os.path.exists('./reranker-finetuned'):
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    shutil.copytree('./reranker-finetuned', save_path)

print(f'Saved to: {save_path}')
print('Done! Download for backend integration.')

In [ ]:
# Step 6: Đánh giá re-ranker vs embedding-only
# Load test data riêng để đánh giá (không dùng train data)
test_path = os.path.join(DRIVE_DIR, 'test.jsonl')
with open(test_path, 'r', encoding='utf-8') as f:
    test_data = [json.loads(line) for line in f]

test_pairs = []
for item in test_data:
    msgs = item['messages']
    for i in range(len(msgs) - 1):
        if msgs[i]['role'] == 'user' and msgs[i+1]['role'] == 'assistant':
            q = msgs[i]['content'].strip()
            a = msgs[i+1]['content'].strip()
            if len(q) > 10 and len(a) > 15:
                test_pairs.append({'query': q, 'answer': a})
print(f'Test pairs: {len(test_pairs)}')

eval_pairs_rr = test_pairs[:100]
eval_queries = [p['query'] for p in eval_pairs_rr]

# Batch encode queries
q_embs = bi_encoder.encode(
    [f'query: {q}' for q in eval_queries],
    normalize_embeddings=True, batch_size=64,
)

# Reuse corpus_embeddings from cell 5
all_docs = corpus

recall_embed_3, recall_embed_5 = 0, 0
recall_rerank_3, recall_rerank_5 = 0, 0

for i, pair in enumerate(eval_pairs_rr):
    target = pair['answer']
    scores = q_embs[i] @ corpus_embeddings.T
    top20_idx = np.argsort(scores)[-20:][::-1]
    top20_docs = [all_docs[j] for j in top20_idx]

    recall_embed_3 += int(target in top20_docs[:3])
    recall_embed_5 += int(target in top20_docs[:5])

    rerank_scores = cross_encoder.predict([(eval_queries[i], doc) for doc in top20_docs])
    rerank_order = np.argsort(rerank_scores)[::-1]
    reranked = [top20_docs[j] for j in rerank_order]

    recall_rerank_3 += int(target in reranked[:3])
    recall_rerank_5 += int(target in reranked[:5])

n = len(eval_pairs_rr)
e3, r3 = recall_embed_3/n, recall_rerank_3/n
e5, r5 = recall_embed_5/n, recall_rerank_5/n

print('='*55)
print('  RE-RANKER IMPACT (trước vs sau re-rank)')
print('='*55)
print(f'{"Metric":>15} | {"Embed only":>11} | {"+ Re-ranker":>11} | {"Delta":>8}')
print('-' * 55)
print(f'{"Recall@3":>15} | {e3:11.4f} | {r3:11.4f} | {r3-e3:+8.4f}')
print(f'{"Recall@5":>15} | {e5:11.4f} | {r5:11.4f} | {r5-e5:+8.4f}')
print()
if r3 > e3:
    print('✅ Re-ranker cải thiện retrieval!')
elif r3 == e3:
    print('➡️ Re-ranker không thay đổi (có thể cần thêm data hoặc epochs)')
else:
    print('⚠️ Re-ranker làm giảm — nên tắt trong production')